In [ ]:
# ==== THAM SỐ (cell duy nhất cần sửa) — đủ 20 CLI arg của src.train ====
REPO_URL = "https://github.com/NguyenMinhTri24072005/MV_CAAFNet.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/MV_CAAFNet"
DRIVE_DATA_SUBDIR = "data/processed"   # đổi thành "data/processed_s1_slice" cho S1
LOCAL_DATA_DIR = "/content/processed"  # --data-dir
OUT_DIR = f"{DRIVE_ROOT}/experiments"  # --out-dir
VERIFY_H5_SHA256 = True

# Kiểm thử (Phase 3, 4, 6)
RUN_PYTEST = False
PYTEST_ARGS = "-q"
LOG_MODEL_PARAMS = False               # sinh reports/model_params.md + đo VRAM batch 16 & tốc độ Hausdorff EDT (P3.7)
SMOKE = False
SMOKE_ALL_CONFIGS = False
OVERFIT_STEPS = None                   # --overfit-steps (ví dụ 150 ở P6.4b)
MAX_BATCHES = None                     # --max-batches

# Tham số run chính (src.train)
CONFIG = "caaf_hausdorff"              # --config configs/{CONFIG}.yaml
DATASET = "luna16"                     # --dataset ("luna16" | "lidc")
SEED = 42                              # --seed
RESUME = False                         # --resume
RUN_NAME = None                        # --run-name
RUN_QUEUE = None                       # list dict [{"config": ..., "dataset": ..., "seed": ...}] để chạy nối tiếp

# 10 cờ tắt còn lại (§6.1.1; None = giữ theo config)
MODEL = None                           # --model
AAF_MODE = None                        # --aaf-mode
LOSS = None                            # --loss
LAMBDA = None                          # --lambda
EPOCHS = None                          # --epochs
BATCH_SIZE = None                      # --batch-size
LR = None                              # --lr
WEIGHT_DECAY = None                    # --weight-decay
AMP = None                             # --amp / --no-amp
AUGMENT = None                         # --augment / --no-augment
EXTRA_SET = []                         # lặp --set

# 10 — Huấn luyện (Phase 3–4 pytest, Phase 6–8)

**Runtime:** Colab Pro **GPU** (A100 / L4 / T4). Chỉ sửa **cell tham số**; mọi tham số CLI của `src.train` (20 arg) ở đó.

Thứ tự gợi ý ở lần đầu: `RUN_PYTEST=True` → `LOG_MODEL_PARAMS=True` → `SMOKE_ALL_CONFIGS=True` → `OVERFIT_STEPS=150`
→ train thật (`CONFIG`, hoặc nhiều run nối tiếp bằng `RUN_QUEUE`). Khi Colab ngắt phiên: đặt `RESUME=True` (hoặc chạy lại
`RUN_QUEUE`, mặc định tự resume) — `last.pt` được ghi atomic mỗi epoch trên Drive.

In [ ]:
# Setup chung: mount Drive → clone/pull repo (GH_TOKEN qua GIT_ASKPASS, không in token) → cài thư viện → in GPU
import json, os, platform, shlex, shutil, subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/MV_CAAFNet"


def _git_env():
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")          # Colab Secrets → không ghi token vào notebook
    except Exception:
        token = None                               # repo public hoặc chưa cấu hình secret
    if token:
        askpass = Path("/content/.git_askpass.sh")
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  Username*) echo "x-access-token" ;;\n  *) printf "%s" "$GH_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GH_TOKEN=token)
    return env


def _git(*args):
    subprocess.run(["git", *args], check=True, env=_git_env())


if Path(REPO_DIR, ".git").is_dir():
    _git("-C", REPO_DIR, "fetch", "--quiet", "origin", BRANCH)
    _git("-C", REPO_DIR, "checkout", "--quiet", BRANCH)
    _git("-C", REPO_DIR, "pull", "--quiet", "--ff-only", "origin", BRANCH)
else:
    _git("clone", "--quiet", "--branch", BRANCH, REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)


def run(args):
    """Chạy `python <args>` từ thư mục repo (danh sách tham số → an toàn với đường dẫn có dấu cách)."""
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=REPO_DIR)


import torch

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"commit {commit} | Python {platform.python_version()} | PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 1024 ** 3:.1f} GB")
else:
    print("⚠ Không có GPU — Runtime → Change runtime type → GPU")

## 1. Kiểm thử `pytest` (Phase 3, 4 — chạy được trước khi có `.h5`)

In [ ]:
if RUN_PYTEST:
    run(["-m", "pytest", "tests", "-p", "no:cacheprovider", "-rs", *shlex.split(PYTEST_ARGS)])

## 2. Dữ liệu `.h5` → `/content/processed` (kiểm SHA-256)

In [ ]:
# Chép .h5 từ Drive về SSD /content (một lần mỗi phiên) và kiểm SHA-256 theo manifest.json
from src.data.pack_h5 import verify_manifest


def stage_dataset(ds, verify=True):
    src = Path(DRIVE_ROOT) / DRIVE_DATA_SUBDIR / ds
    dst = Path(LOCAL_DATA_DIR) / ds
    if not (src / "manifest.json").exists():
        raise FileNotFoundError(f"Chưa có {src}/manifest.json — chạy 01_colab_preprocess.ipynb trước")
    fresh = not ((dst / "manifest.json").exists()
                 and (dst / "manifest.json").read_bytes() == (src / "manifest.json").read_bytes())
    if fresh:
        shutil.rmtree(dst, ignore_errors=True)
        dst.mkdir(parents=True)
        for name in json.loads((src / "manifest.json").read_text(encoding="utf-8"))["files"]:
            shutil.copy2(src / name, dst / name)
        shutil.copy2(src / "manifest.json", dst / "manifest.json")  # chép cuối cùng
    if verify:
        bad = verify_manifest(dst)
        if bad:
            raise RuntimeError(f"SHA-256 sai ở {dst}: {bad} — xóa {dst} rồi chạy lại cell")
    print(f"{ds}: {'đã chép' if fresh else 'đã có sẵn'} → {dst}" + (" | SHA-256 OK" if verify else ""))

datasets = {DATASET} | {job.get("dataset", DATASET) for job in (RUN_QUEUE or [])}
for ds in sorted(datasets):
    stage_dataset(ds, VERIFY_H5_SHA256)

## 3. Số tham số 5 biến thể, VRAM đỉnh batch 16, tốc độ Hausdorff EDT → `reports/model_params.md`

In [ ]:
if LOG_MODEL_PARAMS:
    import time

    from src.data.dataset import H5SegDataset
    from src.losses import BoundaryAwareLoss
    from src.models import build_model, count_parameters
    from src.utils import load_config

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    variants = [("baseline_resunetpp", "ResUNet++ (SE)"), ("pa_resunet", "PA"), ("ca_resunet", "CA"),
                ("pcam_dice", "PCAM tĩnh 0.5/0.5"), ("caaf_hausdorff", "CAAF (AAC thích nghi)")]
    rows = []
    for stem, label in variants:
        cfg = load_config(f"configs/{stem}.yaml")
        model = build_model(cfg).to(device).train()
        info = count_parameters(model)
        amp = cfg["train"]["amp"] if AMP is None else bool(AMP)
        vram = float("nan")
        if device.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            with torch.amp.autocast("cuda", enabled=amp):
                out = model(torch.rand(16, 1, 64, 64, device=device))
            out["prob"].float().mean().backward()
            vram = torch.cuda.max_memory_allocated() / 1024 ** 3
        rows.append((label, stem, info["total"], info["by_module"].get("aac", 0), vram, amp))
        del model
        torch.cuda.empty_cache()

    # Tốc độ Hausdorff EDT: thời gian riêng loss so với cả bước train (batch 16 thật từ train.h5)
    cfg = load_config("configs/caaf_hausdorff.yaml")
    ds = H5SegDataset(Path(LOCAL_DATA_DIR) / DATASET / "train.h5")
    batch = {k: torch.stack([ds[i][k] for i in range(16)]).to(device) for k in ("image", "mask", "dist")}
    model, crit = build_model(cfg).to(device).train(), BoundaryAwareLoss(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)

    def timed(fn, n=20):
        for _ in range(3):
            fn()
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n):
            fn()
        if device.type == "cuda":
            torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n

    def full_step():
        total, _ = crit(model(batch["image"]), batch)
        total.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)

    with torch.no_grad():
        fixed = model(batch["image"])
    t_step = timed(full_step)
    t_loss = timed(lambda: crit(fixed, batch))
    share = t_loss / t_step
    gpu = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
    lines = [
        "# Số tham số & tài nguyên mô hình (Phase 3.7)", "",
        f"Sinh bởi `10_colab_train.ipynb` — commit `{commit}`, {gpu}, PyTorch {torch.__version__}.", "",
        "| Biến thể | Config | Số tham số | Tham số AAC | VRAM đỉnh batch 16 (GB) | AMP |", "|---|---|---|---|---|---|",
        *[f"| {l} | `{s}` | {n:,} | {a:,} | {v:.2f} | {amp} |" for l, s, n, a, v, amp in rows], "",
        "## Tốc độ Hausdorff EDT (batch 16, `caaf_hausdorff`)", "",
        f"- Một bước train đầy đủ: **{1000 * t_step:.1f} ms**; riêng `BoundaryAwareLoss` (gồm EDT trên CPU): "
        f"**{1000 * t_loss:.1f} ms** = **{100 * share:.1f}%**.",
        f"- {'⚠ > 30% → ghi rủi ro R6 (EDT trên CPU là nút thắt)' if share > 0.3 else '≤ 30% → không cần tối ưu EDT'}.",
    ]
    out_md = Path(DRIVE_ROOT) / "reports" / "model_params.md"
    out_md.parent.mkdir(parents=True, exist_ok=True)
    out_md.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("\n".join(lines))

## 4. Dựng danh sách tham số CLI từ cell tham số

In [ ]:
def train_args(config=CONFIG, dataset=DATASET, seed=SEED, run_name=RUN_NAME, resume=RESUME, out_dir=OUT_DIR, extra=()):
    a = ["-m", "src.train", "--config", f"configs/{config}.yaml", "--dataset", dataset,
         "--data-dir", LOCAL_DATA_DIR, "--out-dir", out_dir, "--seed", seed]
    if resume:
        a.append("--resume")
    if run_name:
        a += ["--run-name", run_name]
    if MAX_BATCHES:
        a += ["--max-batches", MAX_BATCHES]
    for flag, value in [("--model", MODEL), ("--aaf-mode", AAF_MODE), ("--loss", LOSS), ("--lambda", LAMBDA),
                        ("--epochs", EPOCHS), ("--batch-size", BATCH_SIZE), ("--lr", LR), ("--weight-decay", WEIGHT_DECAY)]:
        if value is not None:
            a += [flag, value]
    if AMP is not None:
        a.append("--amp" if AMP else "--no-amp")
    if AUGMENT is not None:
        a.append("--augment" if AUGMENT else "--no-augment")
    for item in EXTRA_SET:
        a += ["--set", item]
    return a + list(extra)


print(" ".join(map(str, train_args())))

## 5. Smoke test (`--max-batches 2 --epochs 2` + evaluate), ghi vào `/content/smoke_runs` (không đụng Drive)

In [ ]:
if SMOKE or SMOKE_ALL_CONFIGS:
    configs = (sorted(p.stem for p in Path("configs").glob("*.yaml") if p.stem not in ("base", "data", "paths"))
               if SMOKE_ALL_CONFIGS else [CONFIG])
    smoke_root = "/content/smoke_runs"
    shutil.rmtree(smoke_root, ignore_errors=True)
    for c in configs:
        run(train_args(config=c, run_name=f"smoke_{c}", resume=False, out_dir=smoke_root,
                       extra=["--max-batches", 2, "--epochs", 2]))
        run(["-m", "src.evaluate", "--run-dir", f"{smoke_root}/{DATASET}/smoke_{c}", "--data-dir", LOCAL_DATA_DIR,
             "--no-save-preds"])
    print(f"smoke OK cho {len(configs)} config")

## 6. Overfit 1 batch (P6.4b — kỳ vọng Dice batch > 95%)

In [ ]:
if OVERFIT_STEPS:
    run(train_args(extra=["--overfit-steps", OVERFIT_STEPS]))

## 7. Train thật (một run hoặc `RUN_QUEUE`)

In [ ]:
if OVERFIT_STEPS or SMOKE or SMOKE_ALL_CONFIGS:
    print("Đang ở chế độ kiểm thử (SMOKE / OVERFIT_STEPS) → bỏ qua train thật")
else:
    queue = RUN_QUEUE or [{"config": CONFIG, "dataset": DATASET, "seed": SEED}]
    for i, job in enumerate(queue, 1):
        print(f"=== [{i}/{len(queue)}] {job}", flush=True)
        run(train_args(
            config=job.get("config", CONFIG),
            dataset=job.get("dataset", DATASET),
            seed=job.get("seed", SEED),
            run_name=job.get("run_name", RUN_NAME if not RUN_QUEUE else None),
            resume=job.get("resume", True if RUN_QUEUE else RESUME),   # RUN_QUEUE mặc định tự resume
        ))

## Ghi chú
- Run chuẩn có tên `{config}_s{seed}`; mọi cờ tắt / `EXTRA_SET` làm đổi config sẽ thêm hậu tố `__...` (không đè run chuẩn).
- Đánh giá bằng `11_colab_evaluate.ipynb`; bảng & hình bằng `12_colab_stats_figures.ipynb`.